# 11 — Probability and transition hypotheses

This notebook explains the transition layer rather than presenting another competing possession model. It separates:

`observations → scenario evidence → transition/no-switch hypotheses → interval resolver → optional Viterbi smoothing`

The immediate problem is not just defining more scenario labels. It is deciding when observations are trustworthy enough to change ownership, while preserving a no-switch or contested alternative during flight, poor touch, deflection, crowding, and occlusion.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession_lab').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS = ROOT / 'evaluation' / 'reports'
RESULTS = ROOT / 'evaluation' / 'results'
print('project root:', ROOT)
print('reports:', REPORTS)
print('runner:', ROOT / 'possession_lab' / 'run_interpolated_flight_experiment.py')

## 1. Observation probabilities

For every frame, the feature layer combines control, reception, flight, dead-ball, crowd, contest, missing-ball, distance-margin, and anchor evidence into `[P(H), P(A), P(D), P(X)]`. `X` means uncertain/abstain.

These values are currently a normalized rule-evidence vector produced with a softmax. They are **not calibrated probabilities**: `0.8` does not yet mean 80% empirical correctness. Their role is to keep evidence channels visible and give later decoders bounded emission-like inputs.

In [ ]:
probability_path = REPORTS / 'observation_probability_audit.csv'
columns = [
    'frame', 'scenario_context', 'scenario_state',
    'observation_probability_H', 'observation_probability_A',
    'observation_probability_D', 'observation_probability_X',
    'observation_probability_entropy', 'observation_probability_margin',
    'observation_probability_calibrated', 'observation_probability_model',
    'missing_ball_cause', 'scenario_crowd_score',
    'transition_hypothesis_state',
]
available = pd.read_csv(probability_path, nrows=0).columns
usecols = [column for column in columns if column in available]
probability = pd.read_csv(probability_path, usecols=usecols)
state_cols = [column for column in [
    'observation_probability_H', 'observation_probability_A',
    'observation_probability_D', 'observation_probability_X'
] if column in probability]
probability['probability_sum'] = probability[state_cols].sum(axis=1)
print('rows:', len(probability))
print('maximum normalization error:', float((probability['probability_sum'] - 1.0).abs().max()))
print('calibrated flag:', probability.get('observation_probability_calibrated', pd.Series(dtype=object)).unique())
probability[state_cols + ['observation_probability_entropy', 'observation_probability_margin']].describe()

## 2. Transition hypotheses are interval-level

The transition layer asks two separate questions:

1. Is a transition-like event happening?
2. If so, which team should own the next state?

Each interval keeps explicit alternatives: `H`, `A`, `NO_SWITCH`, and `CONTESTED`/`OCCLUDED`. A kick, direction change, receiver path, control signal, opposite anchor, crowd, deflection risk, or Kalman path can open an interval, but cannot own the next state by itself. This is how we avoid overwriting possession because of one noisy frame between two reliable anchors.

In [ ]:
ledger = pd.read_csv(REPORTS / 'transition_hypothesis_ledger.csv')
ledger_cols = [column for column in ['p_H', 'p_A', 'p_no_switch', 'p_contested'] if column in ledger]
if ledger_cols:
    ledger['hypothesis_probability_sum'] = ledger[ledger_cols].sum(axis=1)
print('transition intervals:', len(ledger))
print(ledger.get('state', pd.Series(dtype=object)).value_counts(dropna=False))
if ledger_cols:
    print('maximum interval normalization error:', float((ledger['hypothesis_probability_sum'] - 1.0).abs().max()))
ledger.head(10)

In [ ]:
if not ledger.empty and ledger_cols:
    view = ledger.sort_values(['activity_max', 'start_frame'], ascending=[False, True]).head(12).sort_values('start_frame')
    ax = view.set_index('start_frame')[ledger_cols].plot.bar(
        figsize=(12, 4), ylim=(0, 1), width=0.85,
        title='High-activity intervals: switch vs no-switch vs contested'
    )
    ax.set_xlabel('interval start frame')
    ax.set_ylabel('rule-derived relative probability')
    plt.tight_layout()
    plt.show()

## 3. Resolver first; Viterbi and HSMM are later layers

The interval resolver is the first ownership decision layer. It requires bounded evidence such as opposing anchors, plausible reception/control, post-contact persistence, and compatible geometry. It can preserve the prior team or abstain.

Viterbi is an optional constrained path smoother after this layer. It may prefer a globally coherent path, but it must not invent evidence; hard anchors and explicit dead-ball constraints remain protected. Contested and occluded intervals make a switch more expensive.

HSMM is not the default yet because we do not have reliable duration distributions for live possession, dead-ball, flight, and contest episodes. Kalman is upstream geometry support only: it estimates a missing-ball path and uncertainty; it does not decide possession.

In [ ]:
candidate_paths = {
    'resolver candidate': RESULTS / 'diagnostic_visible_ball_low_motion_dead_ball.csv',
    'transition-ledger Viterbi': RESULTS / 'baseline_transition_ledger_viterbi.csv',
    'probability-emission Viterbi': RESULTS / 'baseline_probability_transition_ledger_viterbi.csv',
    'duration-aware diagnostic': RESULTS / 'baseline_scenario_confidence_duration.csv',
}
for name, path in candidate_paths.items():
    print(f'{name}: {"available" if path.exists() else "not generated"} — {path.name}')
print('These are decoder variants for evaluation, not independent truth sources.')

## 4. What the probability experiment should solve next

The current model solves transparency and evidence combination. The next probability experiment should solve calibration:

1. reserve reviewed public transition intervals;
2. fit a small calibration layer to the evidence vector and transition outcome;
3. evaluate reliability diagrams and Brier score, not only F1;
4. keep deterministic anchors as constraints;
5. compare resolver-only, Viterbi, and probability-emission Viterbi on identical public windows.

Ground truth remains evaluation-only. The model learns from observable signals and uses forward/backward context only where the intended output permits that latency.